In [1]:
import pandas as pd
import lightgbm as lgb
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

In [2]:
train = pd.read_csv('../001.preprocess/train/train_A.csv')

In [3]:
# 特徴量とターゲットの定義
X = train[['user_id', 'product_id', 'user_interest', 'product_interest', 'event_type']]
y = train['event_type']

In [4]:
# データを訓練用とテスト用に分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# LightGBMのデータセットに変換
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [6]:
# LightGBMモデルの学習
params = {
    'objective': 'multiclass',  # 多クラス分類
    'num_class': len(np.unique(y_train)),  # クラス数はイベントタイプのユニークな数
    'metric': 'multi_logloss',  # ロス関数
    'boosting_type': 'gbdt',  # GBDT (決定木ベース)
    'num_leaves': 31,  # 決定木の葉の数
    'learning_rate': 0.05,  # 学習率
    'feature_fraction': 0.9,  # 特徴量の使用割合
}

model = lgb.train(params, train_data, valid_sets=[test_data], num_boost_round=100, early_stopping_rounds=10)

TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [ ]:
# 予測
y_pred = model.predict(X_test)

In [ ]:
# NDCG評価
# y_predは確率で出力されるので、最大の確率のインデックスを予測順位として使用
y_pred_ranked = np.argsort(-y_pred, axis=1)  # 降順でソート

In [ ]:
# 予測された順位と実際の順位のNDCGを計算
y_true = np.array([np.where(np.array(list(set(y_test))) == label)[0][0] for label in y_test])  # 実際のラベル
ndcg = ndcg_score([y_true], [y_pred_ranked])

print("NDCG Score: ", ndcg)